# PACT-18 两阶段 LoRA · 全量运行（T4×2）

复现公开 0.59x 路线：**stage 1 只用外部语料预训练，stage 2 只用主办方 2,699 条微调**，
骨干截断到前 24 个 block，8 层池化接 plain / centred 双头。

| 项 | 值 |
|---|---|
| 数据 | 2,699 主办方 + 约 4.6 万外部（stage 1 只用外部） |
| 折数 | 3 折 OOF + 1 次全量重训 |
| 预计 | stage 1 ≈ 2.0 h，stage 2 ≈ 0.7 h，经典成员 ≈ 0.7 h，合计 **约 3.5–4 h** |
| 兜底 | `--deadline-hours 8.0`，超出会截断最后一次 fit 而不是让 kernel 被杀 |

提交前请确认 Accelerator = **GPU T4 × 2**、Internet = **Off**。


In [ ]:
import glob, os, sys, time
import torch

print('python  :', sys.version.split()[0])
print('torch   :', torch.__version__, '| cuda', torch.version.cuda)
print('devices :', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  gpu{i}: {p.name} | {p.total_memory / 2**30:.1f} GiB | cc {p.major}.{p.minor}')
print('input   :', sorted(os.listdir('/kaggle/input')))


In [ ]:
import glob, os, sys, time
import json

import torch

print('python  :', sys.version.split()[0])
print('torch   :', torch.__version__, '| cuda', torch.version.cuda)
print('devices :', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  gpu{i}: {p.name} | {p.total_memory / 2**30:.1f} GiB | cc {p.major}.{p.minor}')
print('input   :', sorted(os.listdir('/kaggle/input')))


def pick_dir(patterns, marker):
    for pat in patterns:
        for cand in sorted(glob.glob(pat)):
            if os.path.isdir(cand) and os.path.exists(os.path.join(cand, marker)):
                return cand
    return None

DS = pick_dir(['/kaggle/input/datasets/*/ntl-pact18-verify',
               '/kaggle/input/ntl-pact18-verify',
               '/kaggle/input/*/ntl-pact18-verify'], 'ntl_merged_external_large.parquet')
COMP = pick_dir(['/kaggle/input/competitions/name-that-llm',
                 '/kaggle/input/name-that-llm'], 'test.csv')

# Model source qwen-lm/qwen-3/Transformers/8b-base/1. Explicit single-level
# paths first: each '*' matches one symlinked mount level (a recursive '**'
# scan does not reliably descend into the mount symlinks). The recursive scan
# stays as the fallback - it is what the public 0.55x kernels use.
MODEL = pick_dir([
    '/kaggle/input/models/qwen-lm/qwen-3/Transformers/8b-base/1',
    '/kaggle/input/models/*/qwen-3/Transformers/8b-base/1',
    '/kaggle/input/models/*/*/Transformers/8b-base/1',
    '/kaggle/input/qwen-3',
    '/kaggle/input/*/qwen-3',
], 'config.json')
if MODEL is None:
    for cfg in sorted(glob.glob('/kaggle/input/**/config.json', recursive=True)):
        try:
            meta = json.loads(open(cfg, encoding='utf-8').read())
        except Exception:
            continue
        if str(meta.get('model_type', '')).startswith('qwen3') and int(meta.get('hidden_size', 0)) >= 4096:
            MODEL = os.path.dirname(cfg)
            break
print('DS   :', DS)
print('COMP :', COMP)
print('MODEL:', MODEL)
assert DS and COMP and MODEL, 'Add Input: competition + ntl-pact18-verify + qwen-lm/qwen-3 (Transformers/8b-base/1)'
print('dataset files:', sorted(os.listdir(DS))[:8])

In [ ]:
!pip install -q --no-index --find-links {DS} bitsandbytes 2>&1 | tail -2
import bitsandbytes as bnb
print('bitsandbytes', bnb.__version__)
import transformers
print('transformers', transformers.__version__)


In [ ]:
import shutil, subprocess

WORK = '/kaggle/working/ntl-pact18-code'
if not os.path.isdir(WORK):
    zip_path = f'{DS}/ntl-pact18-code.zip'
    if os.path.exists(zip_path):
        subprocess.call(f'cd /kaggle/working && unzip -oq {zip_path}', shell=True)
    else:
        hits = glob.glob(f'{DS}/**/src/ntl_pact18/pipeline.py', recursive=True)
        assert hits, f'no package under {DS}'
        shutil.copytree(hits[0].split('/src/ntl_pact18/')[0], WORK)
os.chdir(WORK)
sys.path.insert(0, f'{WORK}/src')
print('cwd', os.getcwd())


In [ ]:
!PYTHONPATH={WORK}/src python -m pytest -q tests/test_two_stage.py tests/test_dist_utils.py 2>&1 | tail -3


In [ ]:
# 2,699 host rows + ~46k external rows, with an 'origin' column.
# No oversampling: stage 1 should see every passage once.
rc = subprocess.call(f'''python scripts/build_external_mix.py \\
  --base {COMP}/train.csv \\
  --external {DS}/ntl_merged_external_large.parquet \\
  --target-total 50000 \\
  --out /kaggle/working/train_2stage.csv''', shell=True)
print('build exit', rc)

import pandas as pd
df = pd.read_csv('/kaggle/working/train_2stage.csv')
print('rows', len(df), '| origin', df['origin'].value_counts().to_dict())
print('classes', df['model'].nunique(), '| groups', df['prompt_id'].nunique())
assert rc == 0 and df['model'].nunique() == 18


In [ ]:
t0 = time.time()
rc = subprocess.call(f'''python scripts/train_parallel.py --stage all \\
  --config configs/kaggle_2stage_8b.yaml \\
  --train-file /kaggle/working/train_2stage.csv \\
  --test-file {COMP}/test.csv \\
  --submission /kaggle/working/submission.parquet \\
  --deadline-hours 8.0 --reserve-hours 0.5 \\
  --set features.qwen.model_id={MODEL}''', shell=True)
print('\\n[exit]', rc, 'in', round((time.time() - t0) / 3600, 2), 'h')
assert rc == 0, 'training failed - see artifacts/parallel/logs/qwen_rank0.log'


In [ ]:
import json

meta_path = 'artifacts/parallel/qwen_oof.json'
if os.path.exists(meta_path):
    m = json.load(open(meta_path))
    print('two_stage      :', m.get('two_stage'))
    print('truncate_layers:', m.get('truncate_layers'))
    print('pool_mode      :', m.get('pool_mode'))
    print('layers         :', m.get('layers'))
    print('trainable      :', f"{m.get('trainable_parameters', 0):,}")
    print('per-fit hours  :', [round(s / 3600, 2) for s in m.get('seconds_per_fit', [])])

for f in sorted(glob.glob('artifacts/parallel/logs/*.log')):
    lines = open(f, encoding='utf-8', errors='ignore').readlines()
    print('---', f, '| lines', len(lines))
    print(''.join(lines[-8:]))


In [ ]:
import pyarrow.parquet as pq
from ntl_pact18.labels import LABELS

sub = pd.read_parquet('/kaggle/working/submission.parquet')
test = pd.read_csv(f'{COMP}/test.csv')
print('shape     :', sub.shape)
print('columns   :', list(sub.columns))
print('schema    :', pq.read_schema('/kaggle/working/submission.parquet').names)
print('nulls     :', int(sub.isna().sum().sum()), '| dup ids:', int(sub['id'].duplicated().sum()))
print('labels ok :', set(sub['model']).issubset(set(LABELS)))
print('rows match:', len(sub) == len(test))
assert list(sub.columns) == ['id', 'model']
assert len(sub) == len(test) and sub['id'].is_unique
assert set(sub['model']).issubset(set(LABELS))
print(sub.to_string(index=False))


## 运行结束后怎么读

- `two-stage: pool=... host=2699 ...`：确认 stage 1 / stage 2 的数据确实分开了。
- `truncated trunk: 36 -> 24 blocks`：确认骨干截断。
- `stage1 ... loss`：应在前 50 步内从 ~2.9 降到 2.4 以下；不降说明学习率或数据有问题。
- `stage2: train=1799 ...`：三折各约 1,800 行（2,699 的 2/3）。
- `[qwen] total X.XX h`：总耗时；据此判断是否需要调整 stage 1 的 epoch。
- `submission.parquet`：交互态 3 行；提交后 Kaggle 会换成约 4,000 行重跑。
